# 05 — Chia tập theo nhóm tin cùng căn nhà và đánh giá lại

**Vì sao làm lại.** Tập dữ liệu chia ngẫu nhiên theo dòng (`splits_v2`), nên một căn nhà đăng nhiều lần có thể có tin ở train và tin khác ở validation/test. Trên test, $R^2$ giảm từ 0,699 xuống 0,677 khi bỏ các tin trùng với train (notebook 03, C67). Phản biện yêu cầu chia theo nhóm các tin cùng căn nhà rồi đánh giá lại, cả phần khoảng dự đoán.

**Thế nào là "cùng căn nhà".** Địa chỉ trong dữ liệu không có số nhà, nên không xác định được căn chắc chắn. Hai tin được coi là cùng căn khi có cùng địa chỉ (chuẩn hóa) và cùng diện tích, và:
* cùng giá (tin đăng lại y hệt), **hoặc**
* cùng số tầng, phòng ngủ, phòng tắm (đăng lại nhưng đổi giá; chỉ xét khi cả ba đều có).

Các tin nối với nhau qua bất kỳ điều kiện nào ở trên thuộc cùng một nhóm. Cách gộp rộng hơn (chỉ cần cùng địa chỉ và diện tích) gộp nhầm nhiều căn khác nhau cùng dự án, nên chỉ dùng để kiểm tra độ nhạy ở C07.

**Quy trình.**
1. `splits_v3`: mỗi nhóm thuộc trọn một tập; nhóm nằm ở tập nào nhiều tin nhất trong `splits_v2` thì giữ ở tập đó (hòa thì chọn ngẫu nhiên, seed 42). Nhờ vậy phần lớn các dòng giữ nguyên tập cũ.
2. Cấu hình mô hình **giữ nguyên** như notebook 03 (tham số, đặc trưng, trọng số 0,5/0,5); không tinh chỉnh lại. Chỉ đổi cách chia: CV 3 fold theo nhóm, và phần 15% dùng để chọn số cây (dừng sớm) cũng chia theo nhóm.
3. Fit trên train mới, đo validation, hiệu chỉnh khoảng 80% trên calibration mới bằng đúng phương pháp đã chọn ở notebook 04, rồi đo test một lần. Tập test cũ đã được xem ở notebook 03 (C67); tập test mới gần như cùng các dòng đó, nên kết quả được ghi là "đánh giá lại".
4. Xuất mô hình mới cho ứng dụng: `models/candidates/final_v3_group_2026-10-26/`. Kết quả nằm ở `experiments/group_split/`.

In [1]:
# C01 — Nhóm tin cùng căn nhà và splits_v3
import hashlib
import json
import pickle
import re
import sys
import unicodedata
from datetime import date
from pathlib import Path
from time import perf_counter

import cloudpickle
import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT / "app"))
from core import model_interface  # noqa: E402  cấu hình mô hình lấy từ file đã khóa
from core.reference import normalize_location_name  # noqa: E402

SEED = 42
SPLITS = ["train", "validation", "calibration", "test"]
V2_DIR = PROJECT_ROOT / "data" / "processed" / "splits_v2"
V3_DIR = PROJECT_ROOT / "data" / "processed" / "splits_v3"
OUT = PROJECT_ROOT / "experiments" / "group_split"
OUT.mkdir(parents=True, exist_ok=True)

loaded = model_interface.load_project_model()
assert loaded.model is not None, loaded.problem
locked, locked_manifest = loaded.model, loaded.manifest

raw_address = pd.read_csv(PROJECT_ROOT / "data" / "vietnam_housing_dataset.csv", usecols=["Address"])["Address"]
data = pd.concat(
    [pd.read_csv(V2_DIR / f"{name}.csv").assign(split_v2=name) for name in SPLITS], ignore_index=True
).sort_values("row_id").reset_index(drop=True)
SPLIT_COLUMNS = [c for c in pd.read_csv(V2_DIR / "train.csv", nrows=0).columns]
data["Address"] = raw_address.iloc[data["row_id"].to_numpy()].to_numpy()
data["address_key"] = data["Address"].map(
    lambda t: "" if pd.isna(t) else re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(t))).strip().lower()
)

# Union-find: tin nối nhau nếu cùng địa chỉ + diện tích và (cùng giá hoặc cùng tầng, phòng ngủ, phòng tắm).
parent = np.arange(len(data))


def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i


def link(index_groups):
    for members in index_groups:
        members = list(members)
        for j in members[1:]:
            parent[find(j)] = find(members[0])


link(data.groupby(["address_key", "Area", "Price"]).indices.values())
structure_known = data[["Floors", "Bedrooms", "Bathrooms"]].notna().all(axis=1)
link(data[structure_known].groupby(["address_key", "Area", "Floors", "Bedrooms", "Bathrooms"]).indices.values())
roots = np.array([find(i) for i in range(len(data))])
# Mã nhóm = row_id nhỏ nhất của nhóm, để ổn định khi chạy lại.
data["group_id"] = pd.Series(data["row_id"].to_numpy()).groupby(roots).transform("min").to_numpy()

# Mỗi nhóm về tập đang chứa nhiều tin nhất của nó trong splits_v2.
rng = np.random.default_rng(SEED)
assignment = {}
for group_id, members in data.groupby("group_id"):
    counts = members["split_v2"].value_counts()
    top = sorted(counts[counts == counts.max()].index)
    assignment[group_id] = top[0] if len(top) == 1 else rng.choice(top)
data["split"] = data["group_id"].map(assignment)

group_size = data["group_id"].map(data["group_id"].value_counts())
moved = data["split"] != data["split_v2"]
print(f"{data['group_id'].nunique()} nhóm; {(group_size > 1).mean():.1%} số tin nằm trong nhóm ≥ 2 tin; "
      f"nhóm lớn nhất {group_size.max()} tin")
print(f"Số dòng đổi tập so với splits_v2: {moved.sum()} ({moved.mean():.1%})")
split_table = pd.DataFrame({
    "splits_v2": data["split_v2"].value_counts().reindex(SPLITS),
    "splits_v3": data["split"].value_counts().reindex(SPLITS),
})
split_table["v3_share_%"] = (split_table["splits_v3"] / len(data) * 100).round(1)
display(split_table)
display(pd.crosstab(data["split_v2"], data["split"]).reindex(index=SPLITS, columns=SPLITS))

V3_DIR.mkdir(parents=True, exist_ok=True)
fingerprints = {}
for name in SPLITS:
    path = V3_DIR / f"{name}.csv"
    data.loc[data["split"] == name, SPLIT_COLUMNS].to_csv(path, index=False)
    fingerprints[name] = hashlib.sha256(path.read_bytes()).hexdigest()
data[["row_id", "group_id", "split_v2", "split"]].to_csv(V3_DIR / "split_manifest.csv", index=False)
split_info = {
    "created": str(date.today()),
    "source": "notebooks/05_group_split.ipynb, C01",
    "group_definition": "cùng địa chỉ (NFC, chữ thường, gộp khoảng trắng) và cùng diện tích, và cùng giá "
                        "hoặc cùng số tầng, phòng ngủ, phòng tắm (khi cả ba đều có); nối bắc cầu (union-find)",
    "assignment": "mỗi nhóm về tập chứa nhiều tin nhất của nó trong splits_v2; hòa thì chọn ngẫu nhiên (seed 42)",
    "groups": int(data["group_id"].nunique()),
    "rows_in_multi_listing_groups": float((group_size > 1).mean()),
    "rows_moved_from_v2": int(moved.sum()),
    "rows": {name: int((data["split"] == name).sum()) for name in SPLITS},
    "sha256": fingerprints,
}
(V3_DIR / "split_info.json").write_text(json.dumps(split_info, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")
print("Đã lưu", V3_DIR.relative_to(PROJECT_ROOT))

2026-10-26 09:46:10.831 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-10-26 09:46:10.849 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


25602 nhóm; 24.4% số tin nằm trong nhóm ≥ 2 tin; nhóm lớn nhất 19 tin
Số dòng đổi tập so với splits_v2: 2345 (7.8%)


,splits_v2,splits_v3,v3_share_%
train,18136,18880,62.5
validation,4535,4268,14.1
calibration,3023,2807,9.3
test,4535,4274,14.1


split,train,validation,calibration,test
split_v2,,,,
train,17513,246,153,224
validation,507,3898,37,93
calibration,338,53,2574,58
test,522,71,43,3899


Đã lưu data\processed\splits_v3


In [2]:
# C02 — Kiểm tra: không nhóm nào nằm ở hai tập; tin trùng giữa train và các tập khác
assert data.groupby("group_id")["split"].nunique().eq(1).all(), "có nhóm nằm ở nhiều tập"


def share_matching_train(split, columns, frame):
    train_keys = set(map(tuple, frame.loc[frame[split] == "train", columns].to_numpy()))
    rows = []
    for name in SPLITS[1:]:
        part = frame.loc[frame[split] == name, columns]
        hits = [tuple(r) in train_keys for r in part.to_numpy()]
        rows.append({"tập": name, "rows": len(part), "share_%": round(100 * np.mean(hits), 1)})
    return pd.DataFrame(rows).set_index("tập")


exact = ["address_key", "Area", "Price"]
loose = ["address_key", "Area"]
checks = pd.concat({
    "v2: trùng địa chỉ+diện tích+giá với train": share_matching_train("split_v2", exact, data)["share_%"],
    "v3: trùng địa chỉ+diện tích+giá với train": share_matching_train("split", exact, data)["share_%"],
    "v3: chỉ cùng địa chỉ+diện tích với train": share_matching_train("split", loose, data)["share_%"],
}, axis=1)
display(checks)
assert (checks["v3: trùng địa chỉ+diện tích+giá với train"] == 0).all()

,v2: trùng địa chỉ+diện tích+giá với train,v3: trùng địa chỉ+diện tích+giá với train,v3: chỉ cùng địa chỉ+diện tích với train
tập,,,
validation,10.6,0.0,21.9
calibration,10.7,0.0,21.6
test,10.8,0.0,22.1


## Cấu hình giữ nguyên, chỉ đổi cách chia

Hai thành phần được dựng lại từ file mô hình đã khóa: bước tiền xử lý (`clone`, chưa học gì) và tham số của notebook 03 (lưu trong `manifest.json`). Số cây chọn như notebook 03: tách 15% phần fit để dừng sớm (100 vòng không cải thiện MAE), rồi fit lại trên toàn bộ phần fit với số cây đó. Khác biệt duy nhất: phần 15% này và các fold CV đều chia theo nhóm.

In [3]:
# C03 — Hàm dùng chung
FEATURES = list(locked.features)
WEIGHTS = dict(locked.weights)
COMPONENTS = {
    name: (clone(pipe.named_steps["preprocessing"]), dict(locked_manifest["components"][name]["params"]))
    for name, pipe in locked.components.items()
}
FAMILY = {"XGBoost": "xgb", "LightGBM": "lgb"}
INNER_TEST_SIZE = 0.15
EARLY_STOPPING_ROUNDS = 100
print({name: params["n_estimators"] for name, (_, params) in COMPONENTS.items()}, WEIGHTS)


def split_frame(name):
    return data.loc[data["split"] == name].reset_index(drop=True)


def model_input(frame):
    # 12 cột gốc + Ward/Project/Street tách từ Address, đúng như mô hình đã khóa đọc.
    return locked.prepare(frame[FEATURES + ["Address"]])


def evaluate(y_true, y_pred):
    return {"MAE": mean_absolute_error(y_true, y_pred), "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
            "R2": r2_score(y_true, y_pred)}


def domain_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    ape = np.abs(y_pred - y_true) / y_true
    return {"<=10%": 100 * np.mean(ape <= 0.10), "<=20%": 100 * np.mean(ape <= 0.20),
            "MdAPE": 100 * np.median(ape), ">1 tỷ": 100 * np.mean(np.abs(y_pred - y_true) > 1),
            **evaluate(y_true, y_pred)}


def fit_component(name, X, y, groups):
    preprocessing, params = COMPONENTS[name]
    inner = GroupShuffleSplit(n_splits=1, test_size=INNER_TEST_SIZE, random_state=SEED)
    fit_idx, stop_idx = next(inner.split(X, y, groups))
    selection_pre = clone(preprocessing)
    X_fit = selection_pre.fit_transform(X.iloc[fit_idx], y.iloc[fit_idx])
    X_stop = selection_pre.transform(X.iloc[stop_idx])
    if FAMILY[name] == "xgb":
        selector = XGBRegressor(**params, early_stopping_rounds=EARLY_STOPPING_ROUNDS)
        selector.fit(X_fit, y.iloc[fit_idx], eval_set=[(X_stop, y.iloc[stop_idx])], verbose=False)
        trees = int(selector.best_iteration) + 1
        estimator = XGBRegressor(**{**params, "n_estimators": trees})
    else:
        selector = lgb.LGBMRegressor(**params)
        selector.fit(X_fit, y.iloc[fit_idx], eval_X=X_stop, eval_y=y.iloc[stop_idx], eval_names=["early_stop"],
                     callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, first_metric_only=True, verbose=False)])
        trees = int(selector.best_iteration_)
        estimator = lgb.LGBMRegressor(**{**params, "n_estimators": trees})
    pipeline = Pipeline([("preprocessing", clone(preprocessing)), ("model", estimator)])
    pipeline.fit(X, y)
    return pipeline, trees


def location_key(frame):
    return frame["City"].map(normalize_location_name).astype(str) + " | " + frame["District"].map(normalize_location_name).astype(str)


def rule_predict(fit, apply):
    # Quy tắc môi giới: trung vị đơn giá của quận (thiếu thì của tỉnh) × diện tích.
    ppm2 = fit["Price"] / fit["Area"]
    by_location = ppm2.groupby(location_key(fit)).median()
    by_city = ppm2.groupby(fit["City"].map(normalize_location_name)).median()
    rate = location_key(apply).map(by_location)
    rate = rate.fillna(apply["City"].map(normalize_location_name).map(by_city)).fillna(ppm2.median())
    return (rate * apply["Area"]).to_numpy()


train = split_frame("train")
X_train, y_train, g_train = model_input(train), train["Price"], train["group_id"]
print("train:", X_train.shape)

{'XGBoost': 3000, 'LightGBM': 3000} {'XGBoost': 0.5, 'LightGBM': 0.5}


train: (18880, 15)


In [4]:
# C04 — CV 3 fold theo nhóm trên train (vài phút)
cv = GroupKFold(n_splits=3, shuffle=True, random_state=SEED)
folds = list(cv.split(X_train, y_train, g_train))
for fit_idx, held_idx in folds:
    assert set(g_train.iloc[fit_idx]).isdisjoint(g_train.iloc[held_idx])

oof = pd.DataFrame({"row_id": train["row_id"], "group_id": g_train, "Price": y_train})
fold_rows = []
for fold, (fit_idx, held_idx) in enumerate(folds, start=1):
    for name in COMPONENTS:
        start = perf_counter()
        pipeline, trees = fit_component(name, X_train.iloc[fit_idx], y_train.iloc[fit_idx], g_train.iloc[fit_idx])
        oof.loc[held_idx, name] = pipeline.predict(X_train.iloc[held_idx])
        fold_rows.append({"fold": fold, "model": name, "trees": trees, "seconds": round(perf_counter() - start, 1),
                          **evaluate(y_train.iloc[held_idx], oof.loc[held_idx, name])})
        print(f"fold {fold} {name}: {trees} cây, MAE {fold_rows[-1]['MAE']:.4f}", flush=True)
    oof.loc[held_idx, "Ensemble"] = sum(WEIGHTS[n] * oof.loc[held_idx, n] for n in WEIGHTS)
    fit_part, held_part = train.iloc[fit_idx], train.iloc[held_idx]
    oof.loc[held_idx, "median_baseline"] = fit_part["Price"].median()
    oof.loc[held_idx, "rule_baseline"] = rule_predict(fit_part, held_part)
    for name in ["Ensemble", "median_baseline", "rule_baseline"]:
        fold_rows.append({"fold": fold, "model": name, **evaluate(y_train.iloc[held_idx], oof.loc[held_idx, name])})

cv_folds = pd.DataFrame(fold_rows)
cv_summary = cv_folds.groupby("model", sort=False)[["MAE", "RMSE", "R2"]].agg(["mean", "std"])
cv_summary.columns = [f"{stat}_cv_{metric}" for metric, stat in cv_summary.columns]
cv_summary = cv_summary.reset_index()
cv_folds.to_csv(OUT / "cv_folds.csv", index=False)
cv_summary.to_csv(OUT / "cv_summary.csv", index=False)
oof.to_csv(OUT / "oof_predictions.csv.gz", index=False, compression="gzip", float_format="%.6f")

random_cv = pd.read_csv(PROJECT_ROOT / "experiments" / "ensemble_oof" / "blend_summary.csv").query("xgb_weight == 0.5").iloc[0]
display(cv_summary.round(4))
print(f"So với CV ngẫu nhiên trên splits_v2 (notebook 03): MAE {random_cv['mean_cv_MAE']:.4f}, R² {random_cv['mean_cv_R2']:.4f}")

fold 1 XGBoost: 859 cây, MAE 0.9337


fold 1 LightGBM: 950 cây, MAE 0.9215


fold 2 XGBoost: 780 cây, MAE 0.9449


fold 2 LightGBM: 1713 cây, MAE 0.9497


fold 3 XGBoost: 690 cây, MAE 0.9442


fold 3 LightGBM: 606 cây, MAE 0.9405


,model,mean_cv_MAE,std_cv_MAE,mean_cv_RMSE,std_cv_RMSE,mean_cv_R2,std_cv_R2
0,XGBoost,0.9409,0.0063,1.2748,0.0041,0.6688,0.0018
1,LightGBM,0.9373,0.0144,1.2713,0.0163,0.6706,0.0092
2,Ensemble,0.9295,0.0098,1.2603,0.0092,0.6763,0.0049
3,median_baseline,1.8462,0.0119,2.2156,0.0076,-0.0004,0.0003
4,rule_baseline,1.5114,0.0354,2.3712,0.0977,-0.1476,0.1018


So với CV ngẫu nhiên trên splits_v2 (notebook 03): MAE 0.9153, R² 0.6848


In [5]:
# C05 — Fit trên toàn bộ train mới, đo validation
final_pipelines, final_trees = {}, {}
for name in COMPONENTS:
    final_pipelines[name], final_trees[name] = fit_component(name, X_train, y_train, g_train)
print("Số cây:", final_trees)


def predict_final(X):
    return sum(weight * final_pipelines[name].predict(X) for name, weight in WEIGHTS.items())


validation = split_frame("validation")
X_val, y_val = model_input(validation), validation["Price"]
val_pred = predict_final(X_val)
val_metrics = evaluate(y_val, val_pred)
val_components = {name: evaluate(y_val, final_pipelines[name].predict(X_val)) for name in COMPONENTS}
display(pd.DataFrame({"Ensemble": val_metrics, **val_components}).T.round(4))
print("Validation cũ (splits_v2, notebook 03):", {k: round(v, 4) for k, v in locked_manifest["metrics"]["validation"].items()})

Số cây: {'XGBoost': 1043, 'LightGBM': 683}


,MAE,RMSE,R2
Ensemble,0.8935,1.2243,0.6858
XGBoost,0.8988,1.2359,0.6798
LightGBM,0.9070,1.2355,0.6800


Validation cũ (splits_v2, notebook 03): {'MAE': 0.8802, 'RMSE': 1.21, 'R2': 0.6947}


## Khoảng dự đoán 80% trên calibration mới

Dùng đúng phương pháp đã chọn ở notebook 04: split conformal theo 5 nhóm giá dự đoán, hai đầu riêng, cắt về khoảng giá của train và luôn chứa giá ước tính. Không chọn lại phương pháp; khoảng đều chỉ in ra để tham khảo.

In [6]:
# C06 — Học khoảng trên calibration mới, đo trên validation
ALPHA = 0.2
N_GROUPS = 5
PRICE_RANGE = (float(train["Price"].min()), float(train["Price"].max()))


def order_stat(sorted_values, rank):
    assert 1 <= rank <= len(sorted_values)
    return float(sorted_values[rank - 1])


def fit_grouped(pred, y):
    edges = np.quantile(pred, np.arange(1, N_GROUPS) / N_GROUPS)
    group = np.searchsorted(edges, pred, side="right")
    groups = []
    for g in range(N_GROUPS):
        r = np.sort((y - pred)[group == g])
        n = len(r)
        groups.append({"n": int(n), "q_lo": order_stat(r, int(np.floor((n + 1) * ALPHA / 2))),
                       "q_hi": order_stat(r, int(np.ceil((n + 1) * (1 - ALPHA / 2))))})
    return {"method": "grouped", "edges": [float(e) for e in edges], "groups": groups}


def fit_constant(pred, y):
    scores = np.sort(np.abs(y - pred))
    return {"method": "constant", "q": order_stat(scores, int(np.ceil((len(scores) + 1) * (1 - ALPHA))))}


def interval(params, pred):
    pred = np.asarray(pred, dtype=float)
    if params["method"] == "constant":
        low, high = pred - params["q"], pred + params["q"]
    else:
        g = np.searchsorted(params["edges"], pred, side="right")
        low = pred + np.array([params["groups"][i]["q_lo"] for i in g])
        high = pred + np.array([params["groups"][i]["q_hi"] for i in g])
    low = np.minimum(np.clip(low, *PRICE_RANGE), pred)
    high = np.maximum(np.clip(high, *PRICE_RANGE), pred)
    return low, high


def coverage(params, pred, y, mask=None):
    low, high = interval(params, pred)
    y = np.asarray(y, float)
    inside = (y >= low) & (y <= high)
    g = np.searchsorted(interval_params["edges"], pred, side="right")
    result = {"coverage": float(inside.mean()), "group_coverage": [float(inside[g == i].mean()) for i in range(N_GROUPS)],
              "mean_width": float((high - low).mean()), "median_width": float(np.median(high - low))}
    if mask is not None:
        result["coverage_new_listings"] = float(inside[mask].mean())
    return result


calibration = split_frame("calibration")
cal_pred = predict_final(model_input(calibration))
interval_params = fit_grouped(cal_pred, calibration["Price"].to_numpy())
constant_params = fit_constant(cal_pred, calibration["Price"].to_numpy())

train_loose = set(map(tuple, data.loc[data["split"] == "train", loose].to_numpy()))


def no_loose_match(frame):
    # Căn không có tin nào cùng địa chỉ + diện tích trong train (kiểm tra độ nhạy, cách gộp rộng nhất).
    return ~np.array([tuple(r) in train_loose for r in data.loc[frame["row_id"].map(row_position), loose].to_numpy()])


row_position = pd.Series(np.arange(len(data)), index=data["row_id"])
val_new = no_loose_match(validation)
validation_interval = {
    "grouped": coverage(interval_params, val_pred, y_val, val_new),
    "constant": coverage(constant_params, val_pred, y_val, val_new),
}
display(pd.DataFrame(interval_params["groups"]).assign(
    from_pred=[-np.inf, *interval_params["edges"]], to_pred=[*interval_params["edges"], np.inf]).round(3))
display(pd.DataFrame({k: {m: v for m, v in r.items() if m != "group_coverage"} for k, r in validation_interval.items()}).round(3))
print("Độ phủ theo nhóm (grouped):", [round(c, 3) for c in validation_interval["grouped"]["group_coverage"]])

,n,q_lo,q_hi,from_pred,to_pred
0,562,-1.157,0.762,-inf,4.168
1,561,-1.344,1.465,4.168,5.499
2,561,-1.504,1.656,5.499,6.539
3,560,-1.518,1.982,6.539,7.528
4,563,-1.443,1.399,7.528,inf


,grouped,constant
coverage,0.806,0.805
mean_width,2.847,2.851
median_width,2.842,2.866
coverage_new_listings,0.778,0.779


Độ phủ theo nhóm (grouped): [0.822, 0.804, 0.802, 0.81, 0.794]


In [7]:
# C07 — Đánh giá lại trên test mới (một lần)
test = split_frame("test")
X_test, y_test = model_input(test), test["Price"]
test_pred = predict_final(X_test)
test_new = no_loose_match(test)
test_metrics = evaluate(y_test, test_pred)
test_components = {name: evaluate(y_test, final_pipelines[name].predict(X_test)) for name in COMPONENTS}
test_new_metrics = evaluate(y_test[test_new], test_pred[test_new])
test_domain = pd.DataFrame({
    "median_baseline": domain_metrics(y_test, np.full(len(test), train["Price"].median())),
    "rule_baseline": domain_metrics(y_test, rule_predict(train, test)),
    "Ensemble": domain_metrics(y_test, test_pred),
})
test_interval = coverage(interval_params, test_pred, y_test, test_new)
old_test = json.loads((PROJECT_ROOT / "experiments" / "final_selection" / "test_evaluation.json").read_text(encoding="utf-8"))

display(pd.DataFrame({"Ensemble": test_metrics, **test_components,
                      "Ensemble, căn không cùng địa chỉ+diện tích với train": test_new_metrics}).T.round(4))
print(f"Căn test không có tin cùng địa chỉ + diện tích trong train: {test_new.mean():.1%} ({test_new.sum()} căn)")
display(test_domain.round(3))
print("Khoảng 80% trên test:", {k: (round(v, 3) if not isinstance(v, list) else [round(c, 3) for c in v]) for k, v in test_interval.items()})
print("Test cũ (splits_v2, notebook 03 C67):", {k: round(v, 4) for k, v in old_test["metrics"].items()},
      "| bỏ tin trùng:", {k: round(v, 4) for k, v in old_test["metrics_without_twins"].items()})

,MAE,RMSE,R2
Ensemble,0.9208,1.2475,0.6865
XGBoost,0.9265,1.2585,0.6809
LightGBM,0.9312,1.2580,0.6812
"Ensemble, căn không cùng địa chỉ+diện tích với train",0.9791,1.3176,0.6604


Căn test không có tin cùng địa chỉ + diện tích trong train: 77.9% (3329 căn)


,median_baseline,rule_baseline,Ensemble
<=10%,18.952,27.515,42.092
<=20%,35.073,51.357,70.075
MdAPE,28.049,19.146,12.155
>1 tỷ,69.420,52.433,35.026
MAE,1.865,1.569,0.921
RMSE,2.228,2.513,1.248
R2,-0.000,-0.272,0.686


Khoảng 80% trên test: {'coverage': 0.789, 'group_coverage': [0.813, 0.787, 0.754, 0.816, 0.775], 'mean_width': 2.848, 'median_width': 2.842, 'coverage_new_listings': 0.765}
Test cũ (splits_v2, notebook 03 C67): {'MAE': 0.8917, 'RMSE': 1.2188, 'R2': 0.6986} | bỏ tin trùng: {'MAE': 0.9343, 'RMSE': 1.2659, 'R2': 0.6771}


In [8]:
# C08 — Chẩn đoán cho báo cáo: theo phân khúc, nhóm điều kiện, nhóm giá (out-of-fold theo nhóm)
e = oof.merge(train[["row_id", "City", "District", "Area"]], on="row_id")
e["err"] = e["Ensemble"] - e["Price"]
e["ae"] = e["err"].abs()
e["ape"] = e["ae"] / e["Price"]
e["ppm2"] = e["Price"] / e["Area"] * 1000
city = e["City"].map(normalize_location_name)
hn, hcm = city.eq("Hà Nội"), city.eq("Hồ Chí Minh")
other = ~(hn | hcm)
mid = e["Area"].between(50, 100, inclusive="left")
segments = {"Nhà nhỏ HN <50": hn & (e["Area"] < 50), "Nhà phố HN 50-100": hn & mid,
            "Nhà nhỏ HCM <50": hcm & (e["Area"] < 50), "Nhà phố HCM 50-100": hcm & mid,
            "Nhà lớn HCM >=100": hcm & (e["Area"] >= 100), "Tỉnh khác 50-100": other & mid,
            "Tỉnh khác >=100": other & (e["Area"] >= 100)}
location_rows = location_key(train).map(location_key(train).value_counts()).to_numpy()
optional = ["Frontage", "Access Road", "House direction", "Balcony direction", "Floors", "Bedrooms", "Bathrooms",
            "Legal status", "Furniture state"]
missing = train[optional].isna().sum(axis=1).to_numpy()
conditions = {"quận >=100 tin": location_rows >= 100, "quận 30-99 tin": (location_rows >= 30) & (location_rows < 100),
              "quận <30 tin": location_rows < 30, "thiếu 0-2 trường": missing <= 2,
              "thiếu 3-5 trường": (missing >= 3) & (missing <= 5), "thiếu 6-9 trường": missing >= 6,
              "Area <50": (e["Area"] < 50).to_numpy(), "Area >=200": (e["Area"] >= 200).to_numpy()}


def summarize(masks):
    return pd.DataFrame({k: {"n": int(m.sum()), "price": e.loc[m, "Price"].median(), "ppm2": e.loc[m, "ppm2"].median(),
                             "MAE": e.loc[m, "ae"].mean(), "bias": e.loc[m, "err"].mean(),
                             "MdAPE": 100 * e.loc[m, "ape"].median()} for k, m in masks.items()}).T


segment_table, condition_table = summarize(segments), summarize(conditions)
price_group = pd.cut(e["Price"], [-np.inf, 3, 5, 7, 9, np.inf], right=False, labels=["<3", "3-5", "5-7", "7-9", ">=9"])
price_table = e.groupby(price_group, observed=True).agg(n=("ae", "size"), bias=("err", "mean"), MAE=("ae", "mean"))
oof_domain = pd.DataFrame({name: domain_metrics(e["Price"], e[name]) for name in
                           ["median_baseline", "rule_baseline", "LightGBM", "XGBoost", "Ensemble"]})
display(oof_domain.round(3), segment_table.round(3), condition_table.round(3), price_table.round(3))
print(f"Dự đoán out-of-fold trải từ {e['Ensemble'].min():.2f} đến {e['Ensemble'].max():.2f} tỷ")
oof_domain.to_csv(OUT / "oof_domain_metrics.csv")
test_domain.to_csv(OUT / "test_domain_metrics.csv")
segment_table.to_csv(OUT / "oof_segments.csv")
condition_table.to_csv(OUT / "oof_conditions.csv")
price_table.to_csv(OUT / "oof_price_groups.csv")

,median_baseline,rule_baseline,LightGBM,XGBoost,Ensemble
<=10%,18.946,28.639,41.658,41.435,41.954
<=20%,36.615,52.018,69.243,69.126,69.698
MdAPE,26.875,19.107,12.512,12.580,12.492
>1 tỷ,67.998,51.504,35.689,35.953,35.493
MAE,1.846,1.511,0.937,0.941,0.929
RMSE,2.216,2.372,1.271,1.275,1.260
R2,-0.000,-0.147,0.671,0.669,0.676


,n,price,ppm2,MAE,bias,MdAPE
Nhà nhỏ HN <50,4966.0,5.68,157.500,0.833,0.001,11.494
Nhà phố HN 50-100,1507.0,7.60,129.091,1.129,-0.035,12.455
Nhà nhỏ HCM <50,2104.0,4.65,130.000,0.769,0.028,12.927
Nhà phố HCM 50-100,4328.0,6.59,99.206,0.929,-0.023,10.942
Nhà lớn HCM >=100,828.0,7.70,62.500,1.142,-0.098,11.852
Tỉnh khác 50-100,2668.0,4.30,59.780,0.898,0.000,14.208
Tỉnh khác >=100,2026.0,5.50,37.917,1.116,-0.001,16.255


,n,price,ppm2,MAE,bias,MdAPE
quận >=100 tin,15974.0,6.00,111.303,0.895,-0.025,11.767
quận 30-99 tin,1908.0,4.50,59.231,0.983,0.026,15.675
quận <30 tin,995.0,4.50,38.889,1.378,0.246,23.969
thiếu 0-2 trường,6931.0,5.60,94.231,0.855,-0.019,11.962
thiếu 3-5 trường,8497.0,5.95,110.000,0.924,-0.007,12.325
thiếu 6-9 trường,3452.0,6.10,105.000,1.092,0.026,14.215
Area <50,7425.0,5.35,148.387,0.820,0.021,12.185
Area >=200,434.0,6.30,23.700,1.546,-0.058,21.343


,n,bias,MAE
Price,,,
<3,2128,0.872,0.970
3-5,4726,0.558,0.820
5-7,5892,0.210,0.775
7-9,4276,-0.633,0.906
>=9,1858,-1.681,1.704


Dự đoán out-of-fold trải từ 0.37 đến 10.79 tỷ


In [9]:
# C09 — Mô hình mới học được gì: permutation importance và SHAP trên validation
N_REPEATS = 5
permutation_rng = np.random.default_rng(SEED)
reference_mae = mean_absolute_error(y_val, val_pred)
rows = []
for feature in X_val.columns:
    increases = []
    for _ in range(N_REPEATS):
        permuted = X_val.copy()
        permuted[feature] = permutation_rng.permutation(permuted[feature].to_numpy())
        increases.append(mean_absolute_error(y_val, predict_final(permuted)) - reference_mae)
    rows.append({"feature": feature, "MAE_increase": np.mean(increases), "MAE_increase_std": np.std(increases)})
permutation_table = pd.DataFrame(rows).sort_values("MAE_increase", ascending=False).reset_index(drop=True)


def output_feature_sources(preprocessing):
    sources = []
    for name, transformer, columns in preprocessing.named_steps["columns"].transformers_:
        if transformer == "drop" or len(columns) == 0:
            continue
        if name == "cat":
            for column, categories in zip(columns, transformer.named_steps["onehot"].categories_):
                sources.extend([column] * len(categories))
        else:
            sources.extend(str(f).replace("missingindicator_", "") for f in transformer.get_feature_names_out(columns))
    return sources


def to_dense(matrix):
    return matrix.toarray() if hasattr(matrix, "toarray") else np.asarray(matrix)


shap_parts = []
for name, weight in WEIGHTS.items():
    preprocessing = final_pipelines[name].named_steps["preprocessing"]
    model = final_pipelines[name].named_steps["model"]
    matrix = preprocessing.transform(X_val)
    sources = output_feature_sources(preprocessing)
    if name == "XGBoost":
        contributions = model.get_booster().predict(xgboost.DMatrix(matrix), pred_contribs=True)
    else:
        contributions = model.booster_.predict(matrix, pred_contrib=True)
    contributions = to_dense(contributions)
    assert np.allclose(contributions.sum(axis=1), model.predict(matrix), atol=1e-3)
    shap_parts.append(weight * pd.DataFrame(contributions[:, :-1], columns=sources).T.groupby(level=0).sum().T)
shap_values = sum(shap_parts).fillna(0.0)
shap_table = (shap_values.abs().mean().sort_values(ascending=False).rename("mean_abs_SHAP")
              .reset_index().rename(columns={"index": "feature"}))
permutation_table.to_csv(OUT / "permutation_importance.csv", index=False)
shap_table.to_csv(OUT / "shap_importance.csv", index=False)
display(permutation_table.round(4), shap_table.round(4))

,feature,MAE_increase,MAE_increase_std
0,Area,0.6835,0.0155
1,District,0.5559,0.0047
2,City,0.5126,0.0089
3,Ward,0.2775,0.0018
4,Floors,0.1688,0.0023
5,Access Road,0.1181,0.0055
6,Bathrooms,0.0640,0.0057
7,Project,0.0342,0.0020
8,Street,0.0340,0.0018
9,Bedrooms,0.0229,0.0009


,feature,mean_abs_SHAP
0,Area,0.7228
1,City_District_Ward,0.5490
2,City,0.3333
3,Floors,0.3121
4,District,0.2981
5,City_District_Street,0.2334
6,Bathrooms,0.2249
7,Access Road,0.2209
8,Bedrooms,0.1416
9,City_Project,0.0673


In [10]:
# C10 — Xuất mô hình mới cho ứng dụng và lưu kết quả
EXPORT_DIR = PROJECT_ROOT / "models" / "candidates" / "final_v3_group_2026-10-26"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = EXPORT_DIR / "house_price_ensemble.pkl"
from importlib.metadata import version as package_version  # noqa: E402

versions = {p: package_version(p) for p in ["numpy", "pandas", "scikit-learn", "scipy", "xgboost", "lightgbm", "cloudpickle"]}
assert versions == locked_manifest["versions"], "phiên bản thư viện khác lúc lưu mô hình cũ"
ensemble_cv = cv_summary.set_index("model").loc["Ensemble"]
metrics = {
    "cv_group_3fold": {m: float(ensemble_cv[f"mean_cv_{m}"]) for m in ["MAE", "RMSE", "R2"]},
    "validation": {k: float(v) for k, v in val_metrics.items()},
    "test": {k: float(v) for k, v in test_metrics.items()},
    "test_no_address_area_match": {k: float(v) for k, v in test_new_metrics.items()},
}
metadata = {
    "name": "AIQS house price ensemble (splits_v3, chia theo nhóm tin cùng căn nhà)",
    "created": str(date.today()),
    "source": "notebooks/05_group_split.ipynb, C05 (fit) và C10 (xuất); cấu hình từ notebook 03",
    "target": "Price — tổng giá, tỷ VND (giá rao năm 2024)",
    "trained_on": f"data/processed/splits_v3/train.csv ({len(train)} dòng)",
    "weights": WEIGHTS,
    "trees": final_trees,
    "metrics": metrics,
}
EnsembleClass = type(locked)  # cùng lớp HousePriceEnsemble của notebook 03 (C68)
final_model = EnsembleClass(components=final_pipelines, weights=WEIGHTS, features=FEATURES,
                            address_features=list(locked.address_features), metadata=metadata)
payload = cloudpickle.dumps(final_model)
reloaded = pickle.loads(payload)
assert np.array_equal(reloaded.predict(test[FEATURES + ["Address"]]), test_pred)
MODEL_PATH.write_bytes(payload)
model_sha = hashlib.sha256(payload).hexdigest()

components = {}
for name in WEIGHTS:
    components[name] = {**locked_manifest["components"][name], "n_estimators": int(final_trees[name]),
                        "params": {**locked_manifest["components"][name]["params"], "n_estimators": int(final_trees[name])}}
manifest = {
    "file": MODEL_PATH.name, "sha256": model_sha, "bytes": len(payload), "format": "cloudpickle",
    "class": locked_manifest.get("class"), "input_columns": locked_manifest.get("input_columns"),
    "verified": "nạp lại trong cùng tiến trình cho dự đoán trùng khớp trên test",
    "name": metadata["name"], "created": metadata["created"], "source": metadata["source"],
    "target": metadata["target"], "trained_on": metadata["trained_on"],
    "split": {"dir": "data/processed/splits_v3", **split_info},
    "weights": WEIGHTS, "components": components, "metrics": metrics,
    "acceptance_R2_test_at_least_0_7": bool(test_metrics["R2"] >= 0.7),
    "previous_model": {"path": "models/candidates/final_v2_2026-10-20", "sha256": locked_manifest["sha256"],
                       "note": "cùng cấu hình, chia tập ngẫu nhiên theo dòng (splits_v2)"},
    "versions": versions,
    "prediction_interval": "experiments/group_split/prediction_interval.json",
}
(EXPORT_DIR / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")
(EXPORT_DIR / "requirements-model.txt").write_text("".join(f"{p}=={v}\n" for p, v in versions.items()), encoding="utf-8")

note = ("Tập test mới (splits_v3) gần như cùng các dòng với tập test đã dùng ở notebook 03 (C67); "
        "đây là lần đánh giá lại theo phản biện, cấu hình không đổi.")
interval_record = {
    "created": str(date.today()), "notebook": "notebooks/05_group_split.ipynb",
    "model": {"path": "models/candidates/final_v3_group_2026-10-26/house_price_ensemble.pkl", "sha256": model_sha},
    "target_coverage": 1 - ALPHA, "price_range": list(PRICE_RANGE),
    "fit_on": {"split": "data/processed/splits_v3/calibration.csv", "rows": int(len(calibration))},
    "method": "grouped", "params": interval_params,
    "rules": {"interval": "[pred + q_lo, pred + q_hi] theo 5 nhóm giá dự đoán; cắt về price_range; luôn chứa pred",
              "selection": "phương pháp đã chọn ở notebook 04 (C04), không chọn lại",
              "coverage_new_listings": "căn không có tin cùng địa chỉ + diện tích trong train"},
    "validation": {"rows": int(len(validation)), "new_listing_share": float(val_new.mean()), "results": validation_interval},
    "test": {"rows": int(len(test)), "new_listing_share": float(test_new.mean()), **test_interval, "note": note},
}
(OUT / "prediction_interval.json").write_text(json.dumps(interval_record, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")
test_record = {
    "status": "group_split_reevaluation", "source": "notebooks/05_group_split.ipynb, C07",
    "split": "data/processed/splits_v3 (theo nhóm tin cùng căn nhà)", "model_sha256": model_sha,
    "rows": int(len(test)), "metrics": metrics["test"], "components": test_components,
    "metrics_no_address_area_match": metrics["test_no_address_area_match"], "no_address_area_match_share": float(test_new.mean()),
    "previous_random_split": {"metrics": old_test["metrics"], "metrics_without_twins": old_test["metrics_without_twins"]},
    "acceptance_R2_at_least_0_7": bool(test_metrics["R2"] >= 0.7), "note": note,
}
(OUT / "test_evaluation.json").write_text(json.dumps(test_record, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")
print("Đã lưu", MODEL_PATH.relative_to(PROJECT_ROOT), model_sha[:12], f"{len(payload) / 1e6:.1f} MB")

Đã lưu models\candidates\final_v3_group_2026-10-26\house_price_ensemble.pkl 3d0859cf83ec 13.8 MB


## Kết luận

* **Nhóm tin cùng căn:** 25.602 nhóm; 24,4% số tin nằm trong nhóm từ 2 tin trở lên. Chuyển sang `splits_v3` làm 2.345 dòng (7,8%) đổi tập; tỷ lệ mới 62,5 / 14,1 / 9,3 / 14,1%. Không còn tin nào ở validation, calibration hay test trùng địa chỉ, diện tích và giá với train (trước đây khoảng 10,7%).
* **CV 3 fold theo nhóm** (cấu hình không đổi): MAE 0,930, RMSE 1,260, $R^2$ 0,676, so với 0,915 / 1,243 / 0,685 khi chia ngẫu nhiên. Ensemble vẫn tốt hơn từng mô hình (XGBoost 0,941, LightGBM 0,937).
* **Validation:** MAE 0,894, $R^2$ 0,686 (trước: 0,880 / 0,695).
* **Test (đánh giá lại, cấu hình không đổi):** MAE 0,921, RMSE 1,248, $R^2$ 0,686 (trước: 0,892 / 1,219 / 0,699). Với 77,9% căn test không có tin nào cùng địa chỉ và diện tích trong train (cách gộp rộng nhất), $R^2$ còn 0,660. Mục tiêu $R^2 \geq 0{,}7$ vẫn chưa đạt.
* **Khoảng 80% (cùng phương pháp notebook 04, học lại trên calibration mới):** độ phủ 80,6% trên validation và 78,9% trên test (75,4–81,6% theo nhóm giá); 76,5% với các căn không có tin cùng địa chỉ và diện tích trong train. Độ rộng trung bình 2,85 tỷ. Độ phủ thực tế với tin hoàn toàn mới hơi thấp hơn 80%, nên ứng dụng ghi rõ con số đo được.
* **Diễn giải:** khoảng chênh so với lần chia ngẫu nhiên là phần lợi thế do tin đăng lại; các con số mới gần với việc dùng cho một tin chưa từng thấy. Thứ tự importance không đổi: diện tích, quận, tỉnh, phường/xã, số tầng, đường vào.

Mô hình `models/candidates/final_v3_group_2026-10-26/` thay cho bản v2 trong ứng dụng; kết quả ở `experiments/group_split/`.